# Generate an ATM Pulse and the Detuning Response

## Imports

In [ ]:
from PulseDesigner.qipATM import ATMGate, ATMDetuneResponse

import numpy as np
import matplotlib.pyplot as plt

## Pulse Parameters

Check readme for general description of how these paramters affect the response.

In [ ]:
# Pulse Parameters
pulseTime = 100  # (us) Total time of the pulse
riseTime = 10  # (us) Amount of time the pulse takes to rise to the maximum amplitude
fallTime = 75  # (us) Amount of time the pulse takes to fall to 0 amplitude
maxAmplitude = 1  # (MHZ) Maximum amplitude the pulse will rise too (This should usually be the rabi frequency)
maxFrequency = 1 # (MHz) Maximum frequency offset the pulse will start at
percentRiseGradient = 0.95# 0.95 # Inital slope of the rising amplitude of the pulse
percentFallGradient = 0.04 # Ending slope of the falling amplitude

Run this cell to generate and see the ATM Pulse defined by the above parameters

In [ ]:
# Convert rise and fall gradient to absolute instead of relative values
riseGradient = maxAmplitude * percentRiseGradient / riseTime
fallGradient = maxAmplitude * percentFallGradient / fallTime

# Multiplying this by 2 because the amplitude is taken as the lab frame amplitude but our experimental setups give rotating frame
maxAmplitude = maxAmplitude * 2

testGate = ATMGate(pulseTime,
                   riseTime,
                   fallTime,
                   maxAmplitude,
                   maxFrequency,
                   riseGradient,
                   fallGradient)

testGate.plotPulses(orientation="h", bothFrequencies=True)

## Detuning Response

Running the next 3 cells will test the pulse at different detunings to find the flip probability as a function of detuning

In [ ]:
testResponse = ATMDetuneResponse()  
testResponse.setATM(testGate)

In [ ]:
testDets = np.concat([np.linspace(-1.2, -0.1, 100), np.linspace(-0.1, 0.1, 100), np.linspace(0.1, 1.2, 100)])

testResponse.testDetuning(resolution=1000, detuning=testDets.tolist())
testResponse.updateCharacteristics()

In [ ]:
fig, ax = plt.subplots(1 ,1, figsize=(10, 5))
testResponse.plotReponse(ax)
plt.show()

## Save pulse data

The cells below will save the pulse that was generated above. set the file name to save as in the cell below. (No need to include an extension)

In [ ]:
filename = "atm"

In [ ]:
def fourFiles(
    pulseData: tuple[tuple[list[float], list[float]], tuple[list[float], list[float]]],
    fileLoc: str,
    fileName: str
) -> None:

    for i, side in enumerate(["_left_", "_right_"]):
        for j, iq, in enumerate(["I", "Q"]):
            with open(fileLoc[:-4-len(fileName)] + 
                        fileName + 
                        side + 
                        iq + ".txt", "w") as f:
                for pulseValue in pulseData[i][j]:
                    f.write("{:f}".format(float(np.float16(pulseValue))) + "\n")
    
    with open(fileLoc[:-4-len(fileName)] + 
                fileName + "_params.txt", "w") as f:
        f.write("Pulse Time: {}\n".format(pulseTime))
        f.write("Rise Time: {}\n".format(riseTime))
        f.write("Fall Time: {}\n".format(fallTime))
        f.write("Max Amplitude: {}\n".format(maxAmplitude))
        f.write("Max Frequency: {}\n".format(maxFrequency))
        f.write("Percent Rise Gradient: {}\n".format(percentRiseGradient))
        f.write("Percent Gradient: {}\n".format(percentFallGradient))
        f.write("Rise Gradient: {}\n".format(riseGradient))
        f.write("Fall Gradient: {}".format(fallGradient))
    return

pulseData = testGate.getPulseData()
fourFiles(pulseData, ".", filename)

In [ ]:
times = np.linspace(0, pulseTime, pulseTime*1000)

with open(filename + "_right_I.txt") as f:
    plt.plot(times, [float(val) for val in f.readlines()])

plt.show()
times = np.linspace(0, pulseTime, pulseTime*1000)

with open(filename + "_right_Q.txt") as f:
    plt.plot(times, [float(val) for val in f.readlines()])

plt.show()